# Влияние отвлекающих факторов на успеваемость (ШАГ 0–6), v2

Переделка исследования. **Ключевая находка:** 5 переменных времени суммируются ровно в 24.0 (100% строк) -> композиционные данные, матрица плана сингулярна.

**Корректная спецификация:** одна временная переменная исключается как референсная (`Часы_Сна`); коэффициенты = эффект перераспределения 1 часа **ОТ сна К активности**.

`Средний_Балл` (GPA, шкала ~2.24–4.00) — **ЗАВИСИМАЯ** (y). Остальные — **НЕЗАВИСИМЫЕ** (X).

Графики/таблицы -> `output/`.

**Автор:** Даниил Пикулев


## Импорты и конфигурация

Загружаем библиотеки для данных, визуализации, машинного обучения и статистики. Все константы исследования фиксируются здесь, включая **критерий успеха, зафиксированный ДО выводов**.


In [ ]:
# -*- coding: utf-8 -*-
import os
import platform
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib
# matplotlib.use("Agg")  # закомментировано для интерактивной работы в ноутбуке
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate,
    cross_val_score,
    learning_curve,
    GridSearchCV,
)
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import (
    variance_inflation_factor,
    OLSInfluence,
)
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

RANDOM_STATE = 42
TEST_SIZE = 0.2
N_SPLITS = 5
P_THRESHOLD = 0.05
STABILITY_SEEDS = (0, 20, 42)
HOURS_PER_DAY = 24.0
CLOSURE_TOL = 1e-9
IQR_K = 1.5
PIP_FREEZE_TIMEOUT = 60
# Критерий успеха (бизнес-порог, зафиксирован ДО выводов):
# доля тестовых прогнозов с |ошибка| <= 0.15 балла должна быть >= 80%.
SUCCESS_ABS_TOL = 0.15
SUCCESS_SHARE = 0.80

BASE = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
OUT = os.path.join(BASE, "output")
os.makedirs(OUT, exist_ok=True)
sns.set(style="whitegrid")
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight"})

## Загрузка данных и переименование колонок

Определяем путь к CSV (с запасным вариантом) и приводим названия колонок к русским. Задаём списки числовых переменных, временных (композиционных) колонок, референсную категорию и финальный набор признаков.


In [ ]:
SRC = os.environ.get(
    "STUDENT_LIFESTYLE_CSV",
    os.path.join(BASE, "student_lifestyle_dataset.csv"),
)
if not os.path.exists(SRC):
    # запасной вариант: старый абсолютный путь (для совместимости)
    _legacy_src = (
        "/Users/daniilpikulev/.cache/kagglehub/datasets/"
        "steve1215rogg/student-lifestyle-dataset/versions/1/"
        "student_lifestyle_dataset.csv"
    )
    if os.path.exists(_legacy_src):
        SRC = _legacy_src
if not os.path.exists(SRC):
    sys.exit(
        f"ОШИБКА: CSV не найден: {SRC}\n"
        "Положите файл рядом с analysis.py как 'student_lifestyle_dataset.csv' "
        "или задайте путь через переменную окружения STUDENT_LIFESTYLE_CSV."
    )
RENAME = {
    "Student_ID": "Студенческий_билет_ID",
    "Study_Hours_Per_Day": "Часы_Учебы",
    "Extracurricular_Hours_Per_Day": "Внеклассные_Часы",
    "Sleep_Hours_Per_Day": "Часы_Сна",
    "Social_Hours_Per_Day": "Социальные_Часы",
    "Physical_Activity_Hours_Per_Day": "Часы_Физической_Активности",
    "GPA": "Средний_Балл",
    "Stress_Level": "Уровень_Стресса",
}
NUM = [
    "Часы_Учебы", "Внеклассные_Часы", "Часы_Сна",
    "Социальные_Часы", "Часы_Физической_Активности", "Средний_Балл",
]
TIME5 = NUM[:5]
REF = "Часы_Сна"  # референсная категория (исключена из регрессоров)
FEATS = [
    "Часы_Учебы", "Внеклассные_Часы", "Социальные_Часы",
    "Часы_Физической_Активности", "Уровень_Стресса_ord",
]
YCOL = "Средний_Балл"

df = pd.read_csv(SRC).rename(columns=RENAME)
print(f"shape={df.shape}, n={len(df)}")

## ШАГ 0. Data Quality

Проверяем размер, типы, пропуски, дубликаты, квантили, распределение целевой переменной и уровня стресса. Считаем IQR-флаги выбросов и **проверяем замыкание суммы 5 временных колонок в 24 часа** — ключевой признак композиционности данных.


In [ ]:
print("\n===== ШАГ 0. DATA QUALITY =====")
print(f"строк={df.shape[0]}, колонок={df.shape[1]}")
print(df.dtypes.to_string())
print("\nПропуски:\n" + df.isna().sum().to_string())
print(f"\nДубликаты строк: {df.duplicated().sum()}")
q = [0.001, 0.01, 0.25, 0.5, 0.75, 0.99, 0.999]
print("\nКвантили числовых колонок:\n" + df[NUM].quantile(q).round(4).to_string())
print("\nСредний_Балл: "
      f"mean={df[YCOL].mean():.4f} std={df[YCOL].std():.4f} "
      f"min={df[YCOL].min():.2f} max={df[YCOL].max():.2f} "
      f"ширина={df[YCOL].max() - df[YCOL].min():.2f}")
print("\nУровень_Стресса:\n" + df["Уровень_Стресса"].value_counts().to_string())
print("\nIQR-флаги выбросов:")
for c in NUM:
    q1, q3 = df[c].quantile(0.25), df[c].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - IQR_K * iqr, q3 + IQR_K * iqr
    n = ((df[c] < lo) | (df[c] > hi)).sum()
    print(f"  {c}: Q1={q1:.3f} Q3={q3:.3f} IQR={iqr:.3f} "
          f"границы=[{lo:.3f},{hi:.3f}] выбросов={n} ({n / len(df) * 100:.2f}%)")
s24 = df[TIME5].sum(axis=1)
print("\n*** ПРОВЕРКА ЗАМЫКАНИЯ 24ч: сумма 5 временных колонок: "
      "mean=%.6f min=%.6f max=%.6f" % (s24.mean(), s24.min(), s24.max()))
print(f"*** доля строк с суммой == {HOURS_PER_DAY:.1f} (точность {CLOSURE_TOL:.0e}): "
      f"{(abs(s24 - HOURS_PER_DAY) < CLOSURE_TOL).mean() * 100:.1f}%")
print("*** ВЫВОД: данные композиционные (бюджет 24ч). Все 5 временных переменных + константа")
print(f"*** линейно зависимы -> из регрессоров исключаем референс '{REF}'. Коэффициенты интерпретируются")
print("*** как эффект перераспределения 1 часа ОТ сна К данной активности при фиксированных остальных.")

## ШАГ 0b. Forensic-признаки синтетики

Проверяем числами, похожи ли данные на синтетические: доля значений, кратных 0.1, число уникальных значений и разброс суммы часов. **Перенос результатов на жилых студентов без внешней валидации запрещён.**


In [ ]:
print("\n===== ШАГ 0b. FORENSIC СИНТЕТИКИ =====")
_synth_rows = []
for _c in TIME5 + [YCOL]:
    _s = df[_c]
    _share_01 = float((((_s * 10).round() - (_s * 10)).abs() < 1e-9).mean() * 100)
    _synth_rows.append({"col": _c, "nunique": int(_s.nunique()),
                        "pct_mult_0.1": round(_share_01, 2)})
    print(f"  {_c}: nunique={_s.nunique()}, доля кратных 0.1 = {_share_01:.1f}%")
_s_sum_std = float(s24.std())
print(f"  std(суммы 5 часов) = {_s_sum_std:.3e} (у живых людей с минутами/секундами был бы разброс)")
print("  ВЫВОД: все 5 часовых колонок на 100.0% кратны 0.1 + сумма ровно 24.0 без погрешности -> "
      "признаки округлённой/синтетической генерации; перенос на живых студентов запрещён без внешней валидации.")
pd.DataFrame(_synth_rows).to_csv(os.path.join(OUT, "table_synthetic_forensic.csv"), index=False)

## ШАГ 1. EDA

Разведочный анализ: порядковое кодирование стресса, корреляции Пирсона с целевой переменной, распределения (гистограммы + KDE), боксплот по стрессу, диаграммы рассеяния, тепловая карта корреляций. Также считаем **частную корреляцию** GPA–Стресс при контроле часов учёбы, чтобы показать, что стресс — confounder.


In [ ]:
print("\n===== ШАГ 1. EDA =====")
df["Уровень_Стресса_ord"] = df["Уровень_Стресса"].map({"Low": 0, "Moderate": 1, "High": 2})
corr = df[NUM + ["Уровень_Стресса_ord"]].corr(method="pearson")[YCOL].sort_values(ascending=False)
print("Корреляции Пирсона с Средний_Балл (СВЯЗЬ, НЕ причинность):\n" + corr.round(6).to_string())

fig, axes = plt.subplots(4, 2, figsize=(12, 16))
for ax, c in zip(axes.flat, NUM + ["Уровень_Стресса_ord"]):
    sns.histplot(df[c], kde=True, ax=ax)
    ax.set_title(f"Dist: {c}")
fig.suptitle("Histograms with KDE")
fig.tight_layout()
fig.savefig(os.path.join(OUT, "01_hist_kde.png"))
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="Уровень_Стресса", y=YCOL, order=["Low", "Moderate", "High"], ax=ax)
ax.set_title("Average Grade by Stress_Level")
fig.savefig(os.path.join(OUT, "02_boxplot_stress.png"))
plt.close(fig)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
scatter_cols = [
    "Часы_Учебы", "Внеклассные_Часы", "Часы_Сна",
    "Социальные_Часы", "Часы_Физической_Активности", "Уровень_Стресса_ord",
]
for ax, c in zip(axes.flat, scatter_cols):
    ax.scatter(df[c], df[YCOL], s=6, alpha=0.4)
    ax.set_xlabel(c)
    ax.set_ylabel(YCOL)
    ax.set_title(f"{YCOL} vs {c}")
fig.suptitle("Scatter: y vs each X")
fig.tight_layout()
fig.savefig(os.path.join(OUT, "03_scatter.png"))
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    df[NUM + ["Уровень_Стресса_ord"]].corr(),
    annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax,
)
ax.set_title("Pearson correlation matrix")
fig.savefig(os.path.join(OUT, "04_corr_heatmap.png"))
plt.close(fig)

In [ ]:
r_ys = df[[YCOL, "Уровень_Стресса_ord", "Часы_Учебы"]].corr()
r_y_stress = r_ys.loc[YCOL, "Уровень_Стресса_ord"]
r_y_study = r_ys.loc[YCOL, "Часы_Учебы"]
r_study_stress = r_ys.loc["Уровень_Стресса_ord", "Часы_Учебы"]
pc = (r_y_stress - r_y_study * r_study_stress) / np.sqrt(
    (1 - r_y_study ** 2) * (1 - r_study_stress ** 2)
)
print(f"\nЧастная корреляция r(Средний_Балл, Стресс | Часы_Учебы) = {pc:.4f} "
      f"(парная {corr['Уровень_Стресса_ord']:.4f})")
print(f"r(Часы_Учебы, Стресс) = {r_study_stress:.4f} "
      "-> стресс скоррелирован с учёбой: confounder.")
print("Отрицательные корреляции времени (учёба-физ.акт., сон-физ.акт.) — "
      "артефакт замыкания на 24ч, не субстантивный эффект.")

## ШАГ 2. Кодирование стресса

Сравниваем два способа кодирования уровня стресса: **ordinal 0/1/2** (допускает равную дистанцию и линейный эффект) и **one-hot (drop_first)**. Разница CV-R² ничтожна, поэтому оставляем ordinal как основной (1 степень свободы, проще интерпретация).


In [ ]:
print("\n===== ШАГ 2. КОДИРОВАНИЕ СТРЕССА (в полной ранговой спецификации, без Часы_Сна) =====")
X_ord = df[FEATS]
y = df[YCOL]
base_cols = ["Часы_Учебы", "Внеклассные_Часы", "Социальные_Часы", "Часы_Физической_Активности"]
X_oh = pd.concat(
    [
        df[base_cols],
        pd.get_dummies(df["Уровень_Стресса"], prefix="Stress", drop_first=True, dtype=float),
    ],
    axis=1,
)
cv5 = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for name, XX in [("ordinal 0/1/2 (+допущение равной дистанции)", X_ord), ("one-hot drop_first", X_oh)]:
    s = cross_val_score(LinearRegression(), XX, y, cv=cv5, scoring="r2")
    print(f"  {name}: CV-R2 mean={s.mean():.4f} std={s.std():.4f} "
          f"folds=[{', '.join(f'{v:.4f}' for v in s)}]")
print("Допущение ordinal: равная дистанция Low-Moderate-High, линейный эффект. One-hot его не требует.")
print("Разница CV-R² между кодировками ничтожна -> оставляем ordinal как основной (1 df, проще интерпретация).")

## ШАГ 3. Сравнение моделей

Сплит 80/20 + 5-fold CV. Сравниваем DummyMean, LinearRegression, RidgeCV и RandomForest по RMSE/MAE/R² на тесте и CV. Результаты сохраняются в `table_model_comparison.csv`.


In [ ]:
print("\n===== ШАГ 3. СРАВНЕНИЕ МОДЕЛЕЙ (split 80/20 rs=42 + 5-fold CV) =====")
X = X_ord
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
models = {
    "DummyMean": DummyRegressor(strategy="mean"),
    "LinearRegression": LinearRegression(),
    "RidgeCV": RidgeCV(alphas=np.logspace(-3, 3, 13)),
    "RandomForest": RandomForestRegressor(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
    ),
}
rows = []
for name, m in models.items():
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    rmse = float(np.sqrt(mean_squared_error(y_test, p)))
    cvs = cross_validate(
        m,
        X,
        y,
        cv=cv5,
        scoring=("r2", "neg_root_mean_squared_error", "neg_mean_absolute_error"),
    )
    rows.append({
        "model": name,
        "test_RMSE": rmse,
        "test_MAE": float(mean_absolute_error(y_test, p)),
        "test_R2": float(r2_score(y_test, p)),
        "cv_R2_mean": float(cvs["test_r2"].mean()),
        "cv_R2_std": float(cvs["test_r2"].std()),
        "cv_RMSE_mean": float(-cvs["test_neg_root_mean_squared_error"].mean()),
        "cv_RMSE_std": float(-cvs["test_neg_root_mean_squared_error"].std()),
    })
    print(f"{name}: test RMSE={rmse:.4f} MAE={rows[-1]['test_MAE']:.4f} R2={rows[-1]['test_R2']:.4f} | "
          f"CV R2={rows[-1]['cv_R2_mean']:.4f}±{rows[-1]['cv_R2_std']:.4f} RMSE={rows[-1]['cv_RMSE_mean']:.4f}±{rows[-1]['cv_RMSE_std']:.4f}")
pd.DataFrame(rows).round(4).to_csv(os.path.join(OUT, "table_model_comparison.csv"), index=False)

## ШАГ 3b. Критерий успеха

Критерий зафиксирован заранее: **доля тестовых прогнозов с |ошибка| <= 0.15 балла должна быть >= 80%**. Считаем по каждой модели и строим столбчатую диаграмму.


In [ ]:
# Успех = доля тестовых прогнозов с |ошибка| <= SUCCESS_ABS_TOL (>= SUCCESS_SHARE).
print(f"\n===== ШАГ 3b. КРИТЕРИЙ УСПЕХА: P(|err| <= {SUCCESS_ABS_TOL}) >= {SUCCESS_SHARE:.0%} =====")
success_rows = []
for name, m in models.items():
    _p = m.predict(X_test)
    _share = float((abs(y_test.values - _p) <= SUCCESS_ABS_TOL).mean())
    _ok = _share >= SUCCESS_SHARE
    success_rows.append({"model": name, "tol": SUCCESS_ABS_TOL,
                         "share_le_tol": round(_share, 4), "target": SUCCESS_SHARE,
                         "PASS": "YES" if _ok else "NO"})
    print(f"  {name}: P(|err|<={SUCCESS_ABS_TOL})={_share:.4f} -> {'PASS' if _ok else 'FAIL'}")
pd.DataFrame(success_rows).to_csv(os.path.join(OUT, "table_success_criterion.csv"), index=False)
fig, ax = plt.subplots(figsize=(7, 4))
_sn = pd.DataFrame(success_rows)
ax.bar(_sn["model"], _sn["share_le_tol"])
ax.axhline(SUCCESS_SHARE, color="r", linestyle="--", lw=2, label=f"target {SUCCESS_SHARE:.0%}")
ax.set_ylabel(f"P(|err| <= {SUCCESS_ABS_TOL})")
ax.set_title("Success criterion by model")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT, "10_success_criterion.png"))
plt.close(fig)

## ШАГ 3c. Честный тюнинг RandomForest

GridSearchCV (5-fold, scoring=neg RMSE) для честного сравнения RandomForest с LinearRegression. Сравниваем дефолтную и настроенную модели на тесте.


In [ ]:
print("\n===== ШАГ 3c. ЧЕСТНЫЙ ТЮНИНГ RandomForest (GridSearchCV, 5-fold, scoring=neg RMSE) =====")
_rf_grid = {
    "n_estimators": [200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5],
}
_gs = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    _rf_grid, cv=cv5, scoring="neg_root_mean_squared_error", n_jobs=-1,
)
_gs.fit(X_train, y_train)
print(f"  best_params={_gs.best_params_} best CV RMSE={-float(_gs.best_score_):.4f}")
_tuned = _gs.best_estimator_
_p_def = models["RandomForest"].predict(X_test)
_p_tun = _tuned.predict(X_test)
_tune_rows = [
    {"model": "RF_default(300)", "test_RMSE": round(float(np.sqrt(mean_squared_error(y_test, _p_def))), 4),
     "test_R2": round(float(r2_score(y_test, _p_def)), 4),
     "share_le_0.15": round(float((abs(y_test.values - _p_def) <= SUCCESS_ABS_TOL).mean()), 4)},
    {"model": "RF_tuned", "test_RMSE": round(float(np.sqrt(mean_squared_error(y_test, _p_tun))), 4),
     "test_R2": round(float(r2_score(y_test, _p_tun)), 4),
     "share_le_0.15": round(float((abs(y_test.values - _p_tun) <= SUCCESS_ABS_TOL).mean()), 4)},
]
for _r in _tune_rows:
    print(f"  {_r['model']}: test RMSE={_r['test_RMSE']} R2={_r['test_R2']} P(|err|<=0.15)={_r['share_le_0.15']}")
pd.DataFrame(_tune_rows).to_csv(os.path.join(OUT, "table_rf_tuning.csv"), index=False)
print("  Вывод: сравнение Linear vs RF честно только после тюнинга; "
      "итог см. table_rf_tuning.csv.")
lin = models["LinearRegression"]
y_pred = lin.predict(X_test)
yrange = y.max() - y.min()
print(f"\nИнтерпретация RMSE: линейная модель test RMSE={rows[1]['test_RMSE']:.4f}; "
      f"диапазон y={yrange:.2f} -> "
      f"отн. ошибка {rows[1]['test_RMSE'] / yrange * 100:.1f}%; "
      f"std(y)={y.std():.4f} -> RMSE/std={rows[1]['test_RMSE'] / y.std():.3f}; "
      f"Dummy RMSE={rows[0]['test_RMSE']:.4f} -> "
      f"снижение ошибки {(1 - rows[1]['test_RMSE'] / rows[0]['test_RMSE']) * 100:.1f}%.")

## ШАГ 4. OLS (HC3) + VIF

Оцениваем линейную регрессию через statsmodels с робастными стандартными ошибками HC3. Проверяем **число обусловленности** (полный ранг после исключения референса), считаем **VIF** и таблицу коэффициентов с доверительными интервалами. Сверяем коэффициенты sklearn и OLS.


In [ ]:
print("\n===== ШАГ 4. OLS (HC3) + VIF (полный ранг, референс=Часы_Сна) =====")
Xc = sm.add_constant(X_train, has_constant="add")
print(f"cond(Xc)={np.linalg.cond(Xc.values):.1f} "
      "(полный ранг, было ~3e16 в сингулярной спецификации)")
ols = sm.OLS(y_train, Xc).fit(cov_type="HC3")
print(ols.summary())
vif = pd.DataFrame({
    "feature": X_train.columns,
    "VIF": [
        variance_inflation_factor(Xc.values, i + 1)
        for i in range(X_train.shape[1])
    ],
})
print("\nVIF:\n" + vif.round(3).to_string(index=False))
ci = ols.conf_int()
ci.columns = ["CI_low", "CI_high"]
tab = pd.DataFrame({
    "coef": ols.params,
    "std_err": ols.bse,
    "t": ols.tvalues,
    "p_value": ols.pvalues,
}).join(ci)
tab["VIF"] = tab.index.map(
    dict(zip(["const"] + vif.feature.tolist(), [np.nan] + vif.VIF.tolist()))
)
# p_value НЕ округляем до 5 знаков (иначе 5e-82 превратится в 0.0 в CSV):
# округляем только coef/std_err/CI/VIF, p_value сохраняем полностью.
save_tab = tab.copy()
for _col in ["coef", "std_err", "t", "CI_low", "CI_high", "VIF"]:
    save_tab[_col] = save_tab[_col].round(5)
save_tab.to_csv(os.path.join(OUT, "table_ols_coefficients.csv"))
print("\nТаблица coef:\n" + tab.round(5).to_string())
# сверка sklearn vs OLS
sk = LinearRegression().fit(X_train, y_train)
print(f"\nСверка: max|coef_sklearn - coef_OLS| = "
      f"{np.max(np.abs(sk.coef_ - ols.params[FEATS].values)):.2e} (должно быть ~0)")

### Интерпретация главных коэффициентов

Читаем эффект часов учёбы и стресса после контроля остальных переменных. Парная корреляция стресса исчезает после контроля — признак confounder.


In [ ]:
b_s, p_s = ols.params["Уровень_Стресса_ord"], ols.pvalues["Уровень_Стресса_ord"]
print(f"Главный фактор при контроле остальных: Часы_Учебы "
      f"coef={ols.params['Часы_Учебы']:.4f} "
      f"95%CI=[{ci.loc['Часы_Учебы', 'CI_low']:.4f},{ci.loc['Часы_Учебы', 'CI_high']:.4f}] "
      f"p={ols.pvalues['Часы_Учебы']:.3g}.")
print(f"Стресс после контроля учёбы: coef={b_s:.4f} "
      f"95%CI=[{ci.loc['Уровень_Стресса_ord', 'CI_low']:.4f},"
      f"{ci.loc['Уровень_Стресса_ord', 'CI_high']:.4f}] p={p_s:.3g} -> "
      f"{'ЗНАЧИМ' if p_s < P_THRESHOLD else 'НЕ значим'} на 5%. "
      "Парная r=0.55 исчезает после контроля -> confounder.")

## ШАГ 5. Диагностика OLS

Проверяем гетероскедастичность (Breusch-Pagan), автокорреляцию (Durbin-Watson), строим график y_test vs y_pred, остатки vs fitted, QQ-plot и Cook's distance. Робастные ошибки HC3 оправданы при наличии гетероскедастичности.


In [ ]:
print("\n===== ШАГ 5. ДИАГНОСТИКА OLS =====")
resid = y_test.values - y_pred
bp = het_breuschpagan(ols.resid, ols.model.exog)
print(f"Breusch-Pagan: LM={bp[0]:.3f} p={bp[1]:.3g} -> "
      f"{'гетероскедастичность ЕСТЬ (HC3 оправданы)' if bp[1] < P_THRESHOLD else 'гомоскедастичность не отвергается (HC3 всё равно ок)'}")
print(f"Durbin-Watson={durbin_watson(ols.resid):.3f} (около 2 — автокорреляции нет). "
      f"train R2={ols.rsquared:.4f}, adj={ols.rsquared_adj:.4f}, F p={ols.f_pvalue:.3g}")
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred, s=10, alpha=0.4)
ax.plot([y.min(), y.max()], [y.min(), y.max()], "k--", lw=2)
ax.set_xlabel("y_test")
ax.set_ylabel("y_pred")
ax.set_title("y_test vs y_pred")
fig.savefig(os.path.join(OUT, "05_ytest_vs_ypred.png"))
plt.close(fig)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(lin.predict(X_train), ols.resid, s=8, alpha=0.4)
ax.axhline(0, color="k", lw=1)
ax.set_xlabel("fitted")
ax.set_ylabel("residuals")
ax.set_title("Residuals vs fitted (train)")
fig.savefig(os.path.join(OUT, "06_resid_vs_fitted.png"))
plt.close(fig)
fig = sm.qqplot(ols.resid, line="s")
fig.set_size_inches(7, 5)
fig.suptitle("QQ-plot residuals")
fig.savefig(os.path.join(OUT, "07_qqplot.png"))
plt.close("all")
infl = OLSInfluence(ols)
cooks = infl.cooks_distance[0]
print(f"Cook's distance: max={cooks.max():.5f}, mean={cooks.mean():.6f}, "
      f"точек с D>4/n: {(cooks > 4 / len(X_train)).sum()} из {len(X_train)}")
fig, ax = plt.subplots(figsize=(8, 4))
ax.stem(np.arange(len(cooks)), cooks, markerfmt=" ", basefmt=" ")
ax.set_title("Cook's distance (train)")
fig.savefig(os.path.join(OUT, "08_cooks.png"))
plt.close(fig)

## ШАГ 6. Робастность: интеракция и сегментация

Проверяем интеракцию «Часы учёбы × Стресс» и отдельно оцениваем наклон зависимости GPA от учёбы внутри каждого уровня стресса. Если наклоны близки — эффект учёбы стабилен.


In [ ]:
print("\n===== ШАГ 6. РОБАСТНОСТЬ =====")
Xi = X_train.copy()
Xi["Study_x_Stress"] = Xi["Часы_Учебы"] * Xi["Уровень_Стресса_ord"]
oi = sm.OLS(y_train, sm.add_constant(Xi, has_constant="add")).fit(cov_type="HC3")
print(f"Интеракция Часы_Учебы×Стресс: coef={oi.params['Study_x_Stress']:.5f} "
      f"p={oi.pvalues['Study_x_Stress']:.3g} "
      f"R2={oi.rsquared:.4f} (база {ols.rsquared:.4f})")

In [ ]:
print("\n===== ШАГ 6b. СЕГМЕНТАЦИЯ ПО СТРЕССУ (Low/Moderate/High) =====")
_seg_rows = []
_seg_slopes = {}
for _lvl in ["Low", "Moderate", "High"]:
    _d = df[df["Уровень_Стресса"] == _lvl]
    _X = _d[["Часы_Учебы"]]
    _y = _d[YCOL]
    _m = LinearRegression().fit(_X, _y)
    _pred = _m.predict(_X)
    _r2 = float(r2_score(_y, _pred))
    _rmse = float(np.sqrt(mean_squared_error(_y, _pred)))
    _slope = float(_m.coef_[0])
    _seg_slopes[_lvl] = _slope
    _seg_rows.append({"segment": _lvl, "n": len(_d),
                     "slope_study": round(_slope, 4),
                     "intercept": round(float(_m.intercept_), 4),
                     "R2": round(_r2, 4), "RMSE": round(_rmse, 4)})
    print(f"  {_lvl}: n={len(_d)}, slope(Часы_Учебы)={_slope:.4f}, R2={_r2:.4f}, RMSE={_rmse:.4f}")
_seg = pd.DataFrame(_seg_rows)
_seg.to_csv(os.path.join(OUT, "table_stress_segments.csv"), index=False)
_slopes = list(_seg_slopes.values())
print(f"  Разброс наклонов: min={min(_slopes):.4f} max={max(_slopes):.4f} "
      f"диапазон={max(_slopes) - min(_slopes):.4f}")
print("  Интерпретация: наклоны почти одинаковы (разброс 0.0059) — эффект учёбы стабилен; "
      "но R2 внутри сегментов разный из-за разного разброса данных; "
      "для точного вывода о различии наклонов нужен тест Чоу / интеракция (Шаг 6: p=0.08, нз).")
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, _lvl in zip(axes, ["Low", "Moderate", "High"]):
    _d = df[df["Уровень_Стресса"] == _lvl]
    ax.scatter(_d["Часы_Учебы"], _d[YCOL], s=8, alpha=0.4)
    _xs = np.linspace(_d["Часы_Учебы"].min(), _d["Часы_Учебы"].max(), 50)
    _m = LinearRegression().fit(_d[["Часы_Учебы"]], _d[YCOL])
    ax.plot(_xs, _m.predict(pd.DataFrame({"Часы_Учебы": _xs})), "r-", lw=2)
    ax.set_title(f"{_lvl} (n={len(_d)}, slope={_seg_slopes[_lvl]:.3f})")
    ax.set_xlabel("Часы_Учебы")
axes[0].set_ylabel(YCOL)
fig.suptitle("GPA vs Study by Stress segment")
fig.tight_layout()
fig.savefig(os.path.join(OUT, "11_segments_stress.png"))
plt.close(fig)

## Стабильность сплита и кривая обучения

Проверяем устойчивость результата к разным `random_state` сплита и строим кривую обучения — как меняется R² при росте обучающей выборки.


In [ ]:
print("\nСтабильность сплита:")
for rs in STABILITY_SEEDS:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=rs)
    m = LinearRegression().fit(Xtr, ytr)
    print(f"  random_state={rs}: test R2={r2_score(yte, m.predict(Xte)):.4f} "
          f"RMSE={np.sqrt(mean_squared_error(yte, m.predict(Xte))):.4f}")
tr_sizes, tr_s, te_s = learning_curve(
    LinearRegression(), X, y, cv=cv5,
    train_sizes=np.linspace(0.1, 1.0, 8), scoring="r2",
)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(tr_sizes, tr_s.mean(1), "o-", label="train")
ax.plot(tr_sizes, te_s.mean(1), "o-", label="CV")
ax.fill_between(tr_sizes, te_s.mean(1) - te_s.std(1), te_s.mean(1) + te_s.std(1), alpha=0.2)
ax.set_xlabel("train size")
ax.set_ylabel("R2")
ax.legend()
ax.set_title("Learning curve (LinearRegression)")
fig.savefig(os.path.join(OUT, "09_learning_curve.png"))
plt.close(fig)

## Версии окружения и завершение

Фиксируем версии Python и пакетов (`pip freeze`) для воспроизводимости, сохраняем параметры сплита и seed-ов.


In [ ]:
with open(os.path.join(OUT, "versions.txt"), "w") as f:
    f.write(f"python={platform.python_version()} pandas={pd.__version__}\n")
    try:
        out = subprocess.run(
            [sys.executable, "-m", "pip", "freeze"],
            capture_output=True, text=True, timeout=PIP_FREEZE_TIMEOUT,
        ).stdout
        f.write(out)
    except Exception as e:
        f.write(f"pip freeze failed: {e}\n")
    f.write(f"seed CV={RANDOM_STATE}, split=80/20, "
            f"random_states stability={{{','.join(map(str, STABILITY_SEEDS))}}}\n")
print("\nDONE. Графики и таблицы в output/.")